# Phase 5: Feature Engineering & Hybrid Link Prediction

Until now, our recommender system has been **topology-only**. It only cares about the connections (edges) in the graph. In this notebook, we introduce **Content-Based Features** to build a more robust, 'Hybrid' system.

## 1. The Why: Overcoming the Cold Start Problem
In pure Link Prediction (like our Jaccard baseline), we face the **Cold Start Problem**:
- **New User:** Has no links. Jaccard similarity is 0 with everyone.
- **New Movie:** Has no links. No one can be recommended to it.

By adding node features (User demographics and Movie genres), we can predict links based on **attribute similarity** even when the graph structure is empty.

## 2. The Plan
1. **Vectorization:** Convert categorical data (Gender, Occupation, Genres) into numerical vectors.
2. **Normalization:** Ensure numerical data (Age) is on the same scale (0 to 1).
3. **Similarity Computation:** Use Cosine Similarity to find 'Content Peers'.
4. **Hybrid Scoring:** Merge Graph Similarity + Content Similarity.

## Step 1 & 2: Processing User and Movie Features

### User Features
We use **One-Hot Encoding** for categorical data (Gender, Occupation). This transforms a category like 'Technician' into a binary vector where only the 'Technician' column is 1.

### Movie Features
The MovieLens dataset already provides binary genre flags. We simply extract these into a clean matrix where each row is a movie and each column is a genre.

In [13]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# --- 1. Load and Process User Data ---
# We load demographic info: user_id | age | gender | occupation | zip
users_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
users = pd.read_csv('data/u.user', sep='|', names=users_cols)

# Normalize Age: A 20-year-old and 21-year-old are similar. 
# Without normalization, the 'Age' column would dominate the similarity calculation due to its larger scale.
scaler = MinMaxScaler()
users['age_norm'] = scaler.fit_transform(users[['age']])

# One-Hot Encode Gender and Occupation: 
# This creates a 'sparse' vector for each user representing their identity attributes.
encoder = OneHotEncoder(sparse_output=False)
encoded_features = encoder.fit_transform(users[['gender', 'occupation']])
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(['gender', 'occupation']))

# Combine all features into one 'User Profile' matrix
user_features = pd.concat([users[['user_id', 'age_norm']], encoded_df], axis=1)
user_features.set_index('user_id', inplace=True)

# --- 2. Load and Process Movie Data ---
# MovieLens provides 19 genre flags (Action, Comedy, etc.)
genre_cols = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", 
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", 
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
items_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + genre_cols
items = pd.read_csv('data/u.item', sep='|', names=items_cols, encoding='latin-1')

# Create a 'Movie Profile' matrix based ONLY on genres
movie_features = items[['movie_id'] + genre_cols].copy()
movie_features.set_index('movie_id', inplace=True)

print(f"User Matrix Shape: {user_features.shape} (943 users, each with 24 attributes)")
print(f"Movie Matrix Shape: {movie_features.shape} (1682 movies, each with 19 attributes)")

User Matrix Shape: (943, 24) (943 users, each with 24 attributes)
Movie Matrix Shape: (1682, 19) (1682 movies, each with 19 attributes)


## Step 3: Computing Content Similarity (Cosine Similarity)

Now that we have vectors, we need a way to measure the distance between them. 

### Why Cosine Similarity?
In high-dimensional feature spaces, **Cosine Similarity** is preferred over Euclidean distance. It measures the **angle** between two vectors rather than their magnitude. 
- If two users have identical profiles, their vectors point in the same direction (Similarity = 1.0).
- If they have nothing in common, their vectors are orthogonal (Similarity = 0.0).

In [14]:
# 1. Calculate the N x N User Similarity Matrix
user_sim_matrix = cosine_similarity(user_features)
user_sim_df = pd.DataFrame(user_sim_matrix, index=user_features.index, columns=user_features.index)

# 2. Calculate the M x M Movie Similarity Matrix
movie_sim_matrix = cosine_similarity(movie_features)
movie_sim_df = pd.DataFrame(movie_sim_matrix, index=movie_features.index, columns=movie_features.index)

print("Content-Based Similarity Matrices successfully computed.")
print(f"User 1 vs User 2 Content Similarity: {user_sim_df.loc[1, 2]:.4f}")

Content-Based Similarity Matrices successfully computed.
User 1 vs User 2 Content Similarity: 0.0792


In [15]:
user_sim_df[:5]

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.079211,0.515110,1.000000,0.048083,0.523504,0.518224,0.522922,0.519896,0.520445,...,0.515516,0.523504,0.516054,0.522436,0.523312,0.035742,0.521518,0.511923,0.072064,0.514107
2,0.079211,1.000000,0.074689,0.079211,0.982710,0.155212,0.208743,0.131163,0.101416,0.195419,...,0.221373,0.155212,0.520445,0.177786,0.564967,0.527657,0.114372,0.060981,0.588409,0.070142
3,0.515110,0.074689,1.000000,0.515110,0.045338,0.520759,0.514190,0.520750,0.518429,0.516733,...,0.511176,0.520759,0.515110,0.519150,0.053254,0.033702,0.519745,0.511407,0.067949,0.513377
4,1.000000,0.079211,0.515110,1.000000,0.048083,0.523504,0.518224,0.522922,0.519896,0.520445,...,0.515516,0.523504,0.516054,0.522436,0.523312,0.035742,0.521518,0.511923,0.072064,0.514107
5,0.048083,0.982710,0.045338,0.048083,1.000000,0.094217,0.126711,0.079619,0.061562,0.118623,...,0.134378,0.094217,0.521949,0.107920,0.541690,0.525508,0.069426,0.037017,0.548912,0.042578


## Step 4: The Hybrid Recommender

This is the heart of the notebook. We define a **Hybrid Similarity** score that balances the Graph Structure and the Node Content.

### The Formula
$$ \text{HybridScore} = w \cdot \text{Jaccard}(G) + (1-w) \cdot \text{Cosine}(Features) $$

### Why is this better?
1. **The Graph (Jaccard)** captures **Collaborative Wisdom**: "People who liked this also liked that."
2. **The Content (Cosine)** captures **Individual Context**: "This user is a student, and students often have similar interests."

By setting $w = 0.7$, we give more weight to actual behavior (the graph) but still allow demographic features to influence the ranking.

In [16]:
# 1. Rebuild our Graph structure using the 'u_' and 'm_' prefixes for consistency
df_ratings = pd.read_csv('data/u.data', sep='\t', names=['u_id', 'm_id', 'rating', 'ts'])
pos_df = df_ratings[df_ratings['rating'] >= 4].copy()
pos_df['u_node'] = 'u_' + pos_df['u_id'].astype(str)
pos_df['m_node'] = 'm_' + pos_df['m_id'].astype(str)

G = nx.Graph()
G.add_edges_from(zip(pos_df['u_node'], pos_df['m_node']))

def get_jaccard_similarity(u1, u2, graph):
    """Measures structural similarity in the graph."""
    try:
        m1, m2 = set(graph.neighbors(u1)), set(graph.neighbors(u2))
        return len(m1.intersection(m2)) / len(m1.union(m2)) if len(m1.union(m2)) > 0 else 0.0
    except: return 0.0

def get_hybrid_similarity(u1_node, u2_node, graph, content_sim_df, w_graph=0.7):
    """Calculates a weighted average of Graph and Content similarity."""
    # Convert 'u_1' back to integer 1 for the dataframe lookup
    u1_id = int(u1_node.replace('u_', ''))
    u2_id = int(u2_node.replace('u_', ''))
    
    g_sim = get_jaccard_similarity(u1_node, u2_node, graph)
    c_sim = content_sim_df.loc[u1_id, u2_id]
    
    return (w_graph * g_sim) + ((1 - w_graph) * c_sim)

# Example: Find the most similar peers to User 1 using the Hybrid Score
target_user = 'u_1'
all_user_nodes = [n for n in G.nodes if n.startswith('u_')]

hybrid_peers = []
for other_u in all_user_nodes:
    if other_u == target_user: continue
    score = get_hybrid_similarity(target_user, other_u, G, user_sim_df)
    hybrid_peers.append((other_u, score))

hybrid_peers.sort(key=lambda x: x[1], reverse=True)

print(f"\n--- Top 5 Hybrid Peers for {target_user} ---")
for p_id, score in hybrid_peers[:5]:
    # Find out if they are similar demographically or structurally
    g_score = get_jaccard_similarity(target_user, p_id, G)
    c_score = user_sim_df.loc[int(target_user.replace('u_', '')), int(p_id.replace('u_', ''))]
    print(f"- {p_id}: Total={score:.4f} (Graph={g_score:.4f}, Content={c_score:.4f})")


--- Top 5 Hybrid Peers for u_1 ---
- u_889: Total=0.5056 (Graph=0.2937, Content=1.0000)
- u_738: Total=0.4930 (Graph=0.2784, Content=0.9938)
- u_44: Total=0.4817 (Graph=0.2596, Content=0.9998)
- u_715: Total=0.4593 (Graph=0.2277, Content=0.9995)
- u_244: Total=0.4532 (Graph=0.2192, Content=0.9992)


## Step 5: Testing the Recommender with Detailed Explanations

A great recommender doesn't just give a list of IDs; it provides **Explanations**. This builds trust with the user. 

In this final step, we will:
1. **Aggregate Recommendations:** Find movies liked by the top Hybrid Peers that the target user hasn't seen.
2. **Explain the 'Why':** Show the breakdown of whether a movie came from a 'Behavioral Peer' (someone with the same taste) or a 'Demographic Peer' (someone with a similar background).

In [17]:
def recommend_with_explanations(target_user, G, user_sim_df, top_n=5):
    """
    Generates recommendations by looking at what similar users liked.
    """
    # 1. Get Top 20 Hybrid Peers
    all_user_nodes = [n for n in G.nodes if n.startswith('u_')]
    peers = []
    for other_u in all_user_nodes:
        if other_u == target_user: continue
        score = get_hybrid_similarity(target_user, other_u, G, user_sim_df)
        peers.append((other_u, score))
    
    peers.sort(key=lambda x: x[1], reverse=True)
    top_peers = peers[:20]  # Look at top 20 peers for broader results
    
    # 2. Find movies these peers liked that the user hasn't seen
    user_seen = set(G.neighbors(target_user))
    movie_candidates = Counter()
    movie_evidence = {}  # Store which peer recommended which movie
    
    for peer_id, peer_score in top_peers:
        peer_movies = set(G.neighbors(peer_id))
        for movie in peer_movies:
            if movie not in user_seen:
                movie_candidates[movie] += peer_score
                if movie not in movie_evidence:
                    movie_evidence[movie] = []
                movie_evidence[movie].append(peer_id)

    # 3. Get top recommendations
    recommendations = movie_candidates.most_common(top_n)
    
    print(f"\n=== Hybrid Recommendations for {target_user} ===")
    for m_id, score in recommendations:
        m_idx = int(m_id.replace('m_', ''))
        title = items[items['movie_id'] == m_idx]['title'].values[0]
        
        # Find the primary peer who 'voted' for this movie
        best_peer = movie_evidence[m_id][0]
        g_sim = get_jaccard_similarity(target_user, best_peer, G)
        c_sim = user_sim_df.loc[int(target_user.replace('u_', '')), int(best_peer.replace('u_', ''))]
        
        print(f"\nMovie: {title} (Score: {score:.2f})")
        
        # Logic for the explanation
        if g_sim > 0.1:
            reason = f"Recommended because you share similar taste with User {best_peer.split('_')[1]} (Behavioral Peer)."
        else:
            reason = f"Recommended because User {best_peer.split('_')[1]} has a similar demographic profile to yours (Demographic Peer)."
        
        print(f"- Why: {reason}")
        print(f"- Evidence: Peer {best_peer} similarity breakdown: Graph={g_sim:.2f}, Content={c_sim:.2f}")

# Run the test
recommend_with_explanations('u_1', G, user_sim_df)


=== Hybrid Recommendations for u_1 ===

Movie: Schindler's List (1993) (Score: 5.46)
- Why: Recommended because you share similar taste with User 889 (Behavioral Peer).
- Evidence: Peer u_889 similarity breakdown: Graph=0.29, Content=1.00

Movie: Forrest Gump (1994) (Score: 5.36)
- Why: Recommended because you share similar taste with User 738 (Behavioral Peer).
- Evidence: Peer u_738 similarity breakdown: Graph=0.28, Content=0.99

Movie: One Flew Over the Cuckoo's Nest (1975) (Score: 4.63)
- Why: Recommended because you share similar taste with User 889 (Behavioral Peer).
- Evidence: Peer u_889 similarity breakdown: Graph=0.29, Content=1.00

Movie: Ransom (1996) (Score: 3.80)
- Why: Recommended because you share similar taste with User 488 (Behavioral Peer).
- Evidence: Peer u_488 similarity breakdown: Graph=0.19, Content=0.97

Movie: Apocalypse Now (1979) (Score: 3.78)
- Why: Recommended because you share similar taste with User 889 (Behavioral Peer).
- Evidence: Peer u_889 similari